In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM

import warnings
import massbalancemachine as mbm
import logging
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
import json 

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.utils import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

print(torch.cuda.is_available())   # True
print(torch.cuda.device_count())   # 0  ← wrong$

if torch.cuda.device_count()==0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")


In [ ]:
GLOB_DIR = Path(cfg.dataPath) / path_cache
BASE_DIR = Path(cfg.dataPath) / path_cache / f"TF_REGION"
print(f"Base directory for data: {BASE_DIR}")

# make sure the base directory exists
os.makedirs(BASE_DIR, exist_ok=True)

In [ ]:
MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc")
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

# Cross-regional modelling:

### Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"].head()

In [ ]:
rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

### Monthly datasets:
Build monthly datasets for LSTM. 

In [ ]:
SOURCE_REGIONS = ['CH']
experiment_region_groups = {
    # "CEU": ["FR", "IT_AT", "CH"],
    "HMA": ["CENTRALASIA", "SOUTHASIAWEST", "SOUTHASIAEAST"]
}
paths_multi = {
    "EU_US_CANADA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_EU_US_CANADA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_EU_US_CANADA.nc"),
    },
    "HMA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc"),
    },
}

XREG_PAIRS = [
    # (source, target)
    ("CH", ["ISL"])
]

In [ ]:
# Step 1: collect all unique codes across all pairs
all_codes = sorted(
    {code.upper()
     for src, tgts in XREG_PAIRS
     for code in [src] + tgts})

run_flag_by_code = {
    'CH': False,
}
# Step 2: compute monthly data once per code
monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    region_codes=all_codes,
    run_flag_by_code=run_flag_by_code,
)

# Step 3: assemble train/test pairs from cache — no ERA5 reprocessing
res_xreg_by_source, split_info_by_source = prepare_xreg_pairs_from_cache(
    cfg=cfg,
    monthly_cache=monthly_cache,
    xreg_pairs=XREG_PAIRS,
)

In [ ]:
# EXCLUDE_TARGETS = {"CAW", "ACA", "GRL"}  # set to empty set to include all
EXCLUDE_TARGETS = {}
res_all_xreg_by_source = {}

SOURCE_MEMBERS = {
    "CH": ["CH"],
    "NOR": ["NOR"],
    "ISL": ["ISL"],
    "ALA": ["ALA"],
    "CENTRALASIA": ["CENTRALASIA"],
    "SOUTHASIAWEST": ["SOUTHASIAWEST"],
    "SOUTHASIAEAST": ["SOUTHASIAEAST"],
}

# The 4 individual targets we always want (each source excludes itself)
ALL_TARGETS = [
    "CH", "NOR", "ISL", "ALA", "SJM", "CENTRALASIA", "SOUTHASIAWEST",
    "SOUTHASIAEAST"
]

for src_region in SOURCE_REGIONS:
    src_members = SOURCE_MEMBERS[src_region]

    # Exclude the source region itself from targets
    target_codes = [t for t in ALL_TARGETS if t not in src_members]

    res_all_xreg = build_xreg_res_all(
        res_xreg=res_xreg_by_source[src_region],
        target_source_codes=target_codes,
        source_col="SOURCE_CODE",
        key_prefix=f"XREG_{src_region}_TO",
        ch_code=src_region,
        region_groups={},  # no group regions anymore
    )

    # Filter out excluded targets
    if EXCLUDE_TARGETS:
        res_all_xreg = {
            k: v
            for k, v in res_all_xreg.items()
            if not any(tgt in k for tgt in EXCLUDE_TARGETS)
        }

    res_all_xreg_by_source[src_region] = res_all_xreg
    print(f"{src_region} -> keys:", list(res_all_xreg.keys()))

### Finetuning glaciers:

#### Automatic picking:

In [ ]:
CLIMATE_COLS = [
    't2m', 'tp', 'slhf', 'sshf', 'ssrd', 'fal', 'str', 'ELEVATION_DIFFERENCE'
]
TOPO_COLS = ['aspect', 'slope', 'svf']
# define scalars:
scaler_m, scaler_s, scaler_all = build_global_scalers_multi_source_simple(
    res_xreg_by_source,
    monthly_cols=CLIMATE_COLS,
    static_cols=TOPO_COLS,
)

blur_m, blur_s, blur_joint = estimate_global_bandwidths_simple(
    res_xreg_by_source,
    monthly_cols=CLIMATE_COLS,
    static_cols=TOPO_COLS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    seed=cfg.seed,
)
bandwidths_m = [blur_m * 0.5, blur_m, blur_m * 2.0]
bandwidths_s = [blur_s * 0.5, blur_s, blur_s * 2.0]
print(
    f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
)

In [ ]:
def split_pool_holdout_sinkhorn_v2(
    df_region: pd.DataFrame,
    monthly_cols: list[str],
    static_cols: list[str],
    scaler_m,
    scaler_s,
    blur_joint: float,
    mode: str = "joint",
    topo_extra_cols: list[str] | None = None,
    glacier_col: str = "GLACIER",
    id_col: str = "ID",
    elev_diff_col: str = "ELEVATION_DIFFERENCE",
    pool_frac: float = 0.8,
    seed: int = 0,
    n_restarts: int = 50,
    frac_tol: float = 0.05,
    min_pool_glaciers: int = 3,
    top_k: int = 10,
) -> dict:
    """
    Split a target region into a FT pool and a holdout set at glacier level.
    Returns the top_k best splits (sorted by score ascending) so variability
    across restarts can be assessed.

    Supports two distance modes:
      - "joint": climate + topo features
      - "topo":  topographic features only (static_cols + topo_extra_cols)
    """
    topo_extra_cols = topo_extra_cols or []

    # ------------------------------------------------------------------
    # 1. Glacier inventory
    # ------------------------------------------------------------------
    glaciers = df_region[glacier_col].unique().tolist()
    n_glaciers = len(glaciers)
    glacier_to_idx = {g: i for i, g in enumerate(glaciers)}

    n_meas = np.array(
        [(df_region[glacier_col] == g).sum() for g in glaciers], dtype=int
    )
    n_total_meas     = int(n_meas.sum())
    target_pool_meas = int(round(pool_frac * n_total_meas))

    print(f"  Mode               : {mode.upper()}")
    print(f"  Total measurements : {n_total_meas}")
    print(f"  Target pool meas   : {target_pool_meas} ({pool_frac:.0%})")
    print(f"  Total glaciers     : {n_glaciers}")
    print(f"  top_k              : {top_k}")

    # ------------------------------------------------------------------
    # 2. Feature matrix
    # ------------------------------------------------------------------
    pure_static = [c for c in static_cols if c != elev_diff_col]

    def _stake_topo(df):
        parts = [df.groupby(id_col)[pure_static].first()]
        if elev_diff_col in static_cols:
            parts.append(df.groupby(id_col)[[elev_diff_col]].mean())
        return pd.concat(parts, axis=1)[static_cols].to_numpy(dtype=np.float64)

    id_codes_cat = pd.Categorical(df_region[id_col]).codes
    Xs_id        = scaler_s.transform(_stake_topo(df_region))
    topo_per_row = Xs_id[id_codes_cat]

    if mode == "joint":
        Xm_all = scaler_m.transform(
            df_region[monthly_cols].to_numpy(dtype=np.float64)
        )
        X    = np.hstack([Xm_all, topo_per_row]).astype(np.float32)
        blur = blur_joint
        print(f"  blur_joint         : {blur:.4f}")

    else:  # topo
        extra_parts = []
        for col in topo_extra_cols:
            col_idx  = monthly_cols.index(col)
            col_vals = df_region[col].to_numpy(dtype=np.float64).reshape(-1, 1)
            extra_parts.append(
                (col_vals - scaler_m.mean_[col_idx]) / scaler_m.scale_[col_idx]
            )
        X = np.hstack([topo_per_row] + extra_parts).astype(np.float32) \
            if extra_parts else topo_per_row.astype(np.float32)

        all_topo_cols = static_cols + topo_extra_cols
        n_joint = len(monthly_cols) + len(static_cols)
        n_topo  = len(all_topo_cols)
        blur    = blur_joint * np.sqrt(n_topo / n_joint)
        print(f"  Topo features      : {all_topo_cols}")
        print(f"  blur_topo          : {blur:.4f}")

    glacier_codes = np.array(
        [glacier_to_idx[g] for g in df_region[glacier_col]], dtype=int
    )

    loss_fn = SamplesLoss(
        loss="sinkhorn", p=2, blur=blur,
        scaling=0.9, debias=True, backend="tensorized",
    )

    def _sinkhorn(mask_a, mask_b, max_samples=2000):
        Xa, Xb = X[mask_a], X[mask_b]
        if len(Xa) < 2 or len(Xb) < 2:
            return 0.0
        if len(Xa) > max_samples:
            Xa = Xa[np.random.choice(len(Xa), max_samples, replace=False)]
        if len(Xb) > max_samples:
            Xb = Xb[np.random.choice(len(Xb), max_samples, replace=False)]
        ta = torch.as_tensor(Xa, dtype=torch.float32)
        tb = torch.as_tensor(Xb, dtype=torch.float32)
        wa = torch.ones(len(ta)) / len(ta)
        wb = torch.ones(len(tb)) / len(tb)
        with torch.no_grad():
            return float(loss_fn(wa, ta, wb, tb).item())

    def _mask_for(glacier_idxs):
        return np.isin(glacier_codes, list(glacier_idxs))

    all_mask = np.ones(len(X), dtype=bool)

    # ------------------------------------------------------------------
    # 3. Greedy split — collect ALL valid restarts
    # ------------------------------------------------------------------
    all_results = []

    for restart in range(n_restarts):
        rng   = np.random.default_rng(seed + restart)
        order = np.arange(n_glaciers)
        rng.shuffle(order)

        pool_idxs    = set()
        holdout_idxs = set()
        pool_meas_count = 0

        for glacier_idx in order:
            gl_meas   = int(n_meas[glacier_idx])
            pool_full = pool_meas_count + gl_meas > target_pool_meas * (1 + frac_tol)

            if pool_full:
                holdout_idxs.add(glacier_idx)
                continue

            trial_pool    = pool_idxs    | {glacier_idx}
            trial_holdout = holdout_idxs | {glacier_idx}

            holdout_mask = _mask_for(holdout_idxs) if holdout_idxs else all_mask
            pool_mask    = _mask_for(pool_idxs)    if pool_idxs    else all_mask

            score_if_pool = (
                _sinkhorn(_mask_for(trial_pool), holdout_mask) +
                abs(_sinkhorn(_mask_for(trial_pool), all_mask) -
                    _sinkhorn(holdout_mask, all_mask))
            )
            score_if_holdout = (
                _sinkhorn(pool_mask, _mask_for(trial_holdout)) +
                abs(_sinkhorn(pool_mask, all_mask) -
                    _sinkhorn(_mask_for(trial_holdout), all_mask))
            )

            if score_if_pool <= score_if_holdout:
                pool_idxs.add(glacier_idx)
                pool_meas_count += gl_meas
            else:
                holdout_idxs.add(glacier_idx)

        achieved_frac = pool_meas_count / n_total_meas
        if abs(achieved_frac - pool_frac) > frac_tol:
            continue
        if len(pool_idxs) < min_pool_glaciers:
            continue

        pool_mask    = _mask_for(pool_idxs)
        holdout_mask = _mask_for(holdout_idxs)

        sk_pool_holdout   = _sinkhorn(pool_mask, holdout_mask)
        sk_pool_region    = _sinkhorn(pool_mask, all_mask)
        sk_holdout_region = _sinkhorn(holdout_mask, all_mask)
        score = sk_pool_holdout + abs(sk_pool_region - sk_holdout_region)

        print(f"  restart {restart:3d} | frac={achieved_frac:.2f} | "
              f"sk(pool↔holdout)={sk_pool_holdout:.4f} | "
              f"balance={abs(sk_pool_region - sk_holdout_region):.4f} | "
              f"score={score:.4f}")

        all_results.append((score, {
            "pool_idxs":         pool_idxs,
            "holdout_idxs":      holdout_idxs,
            "pool_meas_count":   pool_meas_count,
            "achieved_frac":     achieved_frac,
            "sk_pool_holdout":   sk_pool_holdout,
            "sk_pool_region":    sk_pool_region,
            "sk_holdout_region": sk_holdout_region,
            "restart":           restart,
            "seed":              seed + restart,
        }))

    if not all_results:
        raise RuntimeError(
            f"No valid split found within frac_tol={frac_tol} and "
            f"min_pool_glaciers={min_pool_glaciers} after {n_restarts} restarts."
        )

    # Sort by score, keep top_k
    all_results.sort(key=lambda x: x[0])
    top_k_results = all_results[:top_k]

    print(f"\n  Valid restarts : {len(all_results)} / {n_restarts}")
    print(f"  Score range    : [{all_results[0][0]:.4f} – {all_results[-1][0]:.4f}]")
    print(f"  Top-{top_k} scores : {[round(r[0], 4) for r in top_k_results]}")

    # ------------------------------------------------------------------
    # 4. Format top-K output
    # ------------------------------------------------------------------
    top_k_splits = []
    for rank, (score, r) in enumerate(top_k_results):
        pool_glaciers    = [glaciers[i] for i in r["pool_idxs"]]
        holdout_glaciers = [glaciers[i] for i in r["holdout_idxs"]]
        top_k_splits.append({
            "rank":            rank,
            "score":           float(score),
            "pool_glaciers":   pool_glaciers,
            "holdout_glaciers": holdout_glaciers,
            "n_glaciers_pool": len(pool_glaciers),
            "n_glaciers_holdout": len(holdout_glaciers),
            "n_meas_pool":     int(r["pool_meas_count"]),
            "n_meas_holdout":  int(n_total_meas - r["pool_meas_count"]),
            "actual_pool_frac": float(r["achieved_frac"]),
            "sk_pool_holdout":   float(r["sk_pool_holdout"]),
            "sk_pool_region":    float(r["sk_pool_region"]),
            "sk_holdout_region": float(r["sk_holdout_region"]),
            "restart":           int(r["restart"]),
            "seed":              int(r["seed"]),
        })

    # Variability summary
    pool_sets    = [frozenset(s["pool_glaciers"]) for s in top_k_splits]
    n_unique     = len(set(pool_sets))
    pairs        = [(pool_sets[i], pool_sets[j])
                    for i in range(len(pool_sets))
                    for j in range(i+1, len(pool_sets))]
    mean_jaccard = np.mean([len(a & b) / len(a | b) for a, b in pairs]) \
                   if pairs else 1.0

    print(f"\n  Top-{top_k} variability:")
    print(f"  Unique pool sets   : {n_unique} / {top_k}")
    print(f"  Mean Jaccard       : {mean_jaccard:.3f}  "
          f"(1.0 = identical, 0.0 = no overlap)")

    return {
        "top_k":       top_k_splits,
        "best":        top_k_splits[0],   # convenience shortcut
        "n_valid":     len(all_results),
        "score_range": (float(all_results[0][0]), float(all_results[-1][0])),
        "n_unique_pool_sets": n_unique,
        "mean_jaccard":       float(mean_jaccard),
        "mode":        mode,
        "blur":        float(blur),
    }

#### FT Pool & test set:

In [ ]:
# ── Run both modes and compare ────────────────────────────────────────────
ISL_SPLIT_PATH_JOINT = BASE_DIR / "isl_test_pool_split_joint.json"
ISL_SPLIT_PATH_TOPO  = BASE_DIR / "isl_test_pool_split_topo.json"
RECOMPUTE_ISL_SPLIT  = True

df_isl_all = res_xreg_by_source["CH"]["df_test"]
df_isl_all = df_isl_all[df_isl_all["SOURCE_CODE"] == "ISL"].copy()

# ── Joint ─────────────────────────────────────────────────────────────────
if not RECOMPUTE_ISL_SPLIT and ISL_SPLIT_PATH_JOINT.exists():
    with open(ISL_SPLIT_PATH_JOINT) as f:
        isl_split_joint = json.load(f)
    print(f"Loaded joint split from {ISL_SPLIT_PATH_JOINT}")
else:
    print("=" * 55)
    print("Computing JOINT pool/test split...")
    isl_split_joint = split_pool_holdout_sinkhorn_v2(
        df_region=df_isl_all,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        scaler_m=scaler_m, scaler_s=scaler_s,
        blur_joint=blur_joint,
        mode="joint",
        pool_frac=0.8, n_restarts=50, seed=cfg.seed,
    )
    with open(ISL_SPLIT_PATH_JOINT, "w") as f:
        json.dump(isl_split_joint, f, indent=2)
    print(f"Saved to {ISL_SPLIT_PATH_JOINT}")

# ── Topo ──────────────────────────────────────────────────────────────────
if not RECOMPUTE_ISL_SPLIT and ISL_SPLIT_PATH_TOPO.exists():
    with open(ISL_SPLIT_PATH_TOPO) as f:
        isl_split_topo = json.load(f)
    print(f"Loaded topo split from {ISL_SPLIT_PATH_TOPO}")
else:
    print("=" * 55)
    print("Computing TOPO pool/test split...")
    isl_split_topo = split_pool_holdout_sinkhorn_v2(
        df_region=df_isl_all,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        scaler_m=scaler_m, scaler_s=scaler_s,
        blur_joint=blur_joint,
        mode="topo",
        topo_extra_cols=["ELEVATION_DIFFERENCE"],
        pool_frac=0.8, n_restarts=50, seed=cfg.seed,
    )
    with open(ISL_SPLIT_PATH_TOPO, "w") as f:
        json.dump(isl_split_topo, f, indent=2)
    print(f"Saved to {ISL_SPLIT_PATH_TOPO}")

# ── Extract pool/test sets — use best (rank-0) split from top_k ──────────
ISL_FT_POOL_JOINT  = isl_split_joint["best"]["pool_glaciers"]
ISL_TEST_SET_JOINT = isl_split_joint["best"]["holdout_glaciers"]
ISL_FT_POOL_TOPO   = isl_split_topo["best"]["pool_glaciers"]
ISL_TEST_SET_TOPO  = isl_split_topo["best"]["holdout_glaciers"]

df_isl_pool_joint = df_isl_all[df_isl_all["GLACIER"].isin(ISL_FT_POOL_JOINT)]
df_isl_test_joint = df_isl_all[df_isl_all["GLACIER"].isin(ISL_TEST_SET_JOINT)]
df_isl_pool_topo  = df_isl_all[df_isl_all["GLACIER"].isin(ISL_FT_POOL_TOPO)]
df_isl_test_topo  = df_isl_all[df_isl_all["GLACIER"].isin(ISL_TEST_SET_TOPO)]

# ── Summary comparison ────────────────────────────────────────────────────
print("\nPool/test split comparison:")
print(f"{'':25} {'JOINT':>20} {'TOPO':>20}")
print(f"{'Pool glaciers':25} {len(ISL_FT_POOL_JOINT):>20} {len(ISL_FT_POOL_TOPO):>20}")
print(f"{'Pool measurements':25} {len(df_isl_pool_joint):>20} {len(df_isl_pool_topo):>20}")
print(f"{'Test glaciers':25} {len(ISL_TEST_SET_JOINT):>20} {len(ISL_TEST_SET_TOPO):>20}")
print(f"{'Test measurements':25} {len(df_isl_test_joint):>20} {len(df_isl_test_topo):>20}")
print(f"{'Best score':25} {isl_split_joint['best']['score']:>20.4f} "
      f"{isl_split_topo['best']['score']:>20.4f}")
print(f"{'Sk(pool→test)':25} {isl_split_joint['best']['sk_pool_holdout']:>20.4f} "
      f"{isl_split_topo['best']['sk_pool_holdout']:>20.4f}")
print(f"{'Balance':25} "
      f"{abs(isl_split_joint['best']['sk_pool_region'] - isl_split_joint['best']['sk_holdout_region']):>20.4f} "
      f"{abs(isl_split_topo['best']['sk_pool_region'] - isl_split_topo['best']['sk_holdout_region']):>20.4f}")
print(f"{'Valid restarts':25} {isl_split_joint['n_valid']:>20} "
      f"{isl_split_topo['n_valid']:>20}")
print(f"{'Score range':25} "
      f"  [{isl_split_joint['score_range'][0]:.4f}–{isl_split_joint['score_range'][1]:.4f}]"
      f"  [{isl_split_topo['score_range'][0]:.4f}–{isl_split_topo['score_range'][1]:.4f}]")
print(f"{'Mean Jaccard top-K':25} {isl_split_joint['mean_jaccard']:>20.3f} "
      f"{isl_split_topo['mean_jaccard']:>20.3f}")
print(f"{'Unique pool sets':25} {isl_split_joint['n_unique_pool_sets']:>20} "
      f"{isl_split_topo['n_unique_pool_sets']:>20}")

# ── Glacier overlap between joint and topo best splits ────────────────────
pool_overlap = set(ISL_FT_POOL_JOINT) & set(ISL_FT_POOL_TOPO)
print(f"\nGlaciers in both pool sets : {len(pool_overlap)} / "
      f"{len(set(ISL_FT_POOL_JOINT) | set(ISL_FT_POOL_TOPO))} unique")
print(f"Pool Jaccard similarity    : "
      f"{len(pool_overlap) / len(set(ISL_FT_POOL_JOINT) | set(ISL_FT_POOL_TOPO)):.3f}")

# ── Top-K variability detail ──────────────────────────────────────────────
for mode_label, isl_split in [("JOINT", isl_split_joint), ("TOPO", isl_split_topo)]:
    print(f"\n{mode_label} top-{len(isl_split['top_k'])} splits:")
    print(f"  {'rank':>4}  {'score':>8}  {'n_pool_gl':>10}  {'pool_glaciers'}")
    for s in isl_split["top_k"]:
        print(f"  {s['rank']:>4}  {s['score']:>8.4f}  "
              f"{s['n_glaciers_pool']:>10}  {sorted(s['pool_glaciers'])}")

In [ ]:
from matplotlib.ticker import FuncFormatter
import scipy.stats as scipy_stats


def format_axis_ticks(ax, label_size=8):
    """Format tick labels to avoid huge 1e6/1e7 offset labels."""
    # check if scientific notation offset is being used
    ax.xaxis.get_major_formatter().set_useOffset(False)
    try:
        # scale large numbers to readable units
        xmax = abs(ax.get_xlim()[1])
        if xmax > 1e6:
            scale = 1e6
            ax.xaxis.set_major_formatter(
                FuncFormatter(lambda x, _: f"{x/scale:.1f}"))
            ax.set_xlabel(f"(×10⁶)", label_size=label_size, labelpad=1)
        elif xmax > 1e4:
            scale = 1e3
            ax.xaxis.set_major_formatter(
                FuncFormatter(lambda x, _: f"{x/scale:.0f}"))
            ax.set_xlabel(f"(×10³)", label_size=label_size, labelpad=1)
    except Exception:
        pass


def plot_kde_pair(glaciers_to_plot, selected_cols, save_prefix):
    """KDE panels for a pair of glaciers."""
    ncols = 3
    nrows = int(np.ceil(len(selected_cols) / ncols))
    w, h = nature_figsize(cols=1, height_mm=160)
    fig, axes = plt.subplots(nrows, ncols, figsize=(w * 2, h), squeeze=False)

    legend_handles = []

    for idx, col in enumerate(selected_cols):
        ax = axes[idx // ncols][idx % ncols]

        all_vals = pd.concat([
            cfg_gl["df"][col].dropna() for cfg_gl in glaciers_to_plot.values()
        ])
        x_grid = np.linspace(float(all_vals.min()), float(all_vals.max()), 500)

        for label, cfg_gl in glaciers_to_plot.items():
            vals = cfg_gl["df"][col].dropna().values
            if len(vals) < 10:
                continue
            kde = scipy_stats.gaussian_kde(vals, bw_method=0.3)
            y = kde(x_grid)
            y = y / y.max()
            line, = ax.plot(x_grid,
                            y,
                            color=cfg_gl["color"],
                            linewidth=0.8,
                            label=label)
            ax.fill_between(x_grid, y, alpha=0.08, color=cfg_gl["color"])

            # collect handles only from first panel to avoid duplicates
            if idx == 0:
                legend_handles.append(line)

        ax.set_title(col, fontsize=8)
        ax.set_ylim(0, 1.15)
        ax.set_xlabel("")
        ax.tick_params(labelsize=8, width=0.4, length=2, direction="in")
        ax.spines[["top", "right", "left", "bottom"]].set_visible(True)
        for spine in ax.spines.values():
            spine.set_linewidth(0.4)
        ax.grid(axis="x", color="#e0e0e0", linewidth=0.3)
        ax.set_axisbelow(True)
        format_axis_ticks(ax, label_size=6)

    # ── place legend in first empty axis if one exists, else above figure ──
    empty_axes = [
        axes[idx // ncols][idx % ncols]
        for idx in range(len(selected_cols), nrows * ncols)
    ]

    if empty_axes:
        leg_ax = empty_axes[0]
        leg_ax.axis("off")
        leg_ax.legend(
            handles=legend_handles,
            loc="center",
            fontsize=8,
            frameon=False,
        )
        # turn off remaining empty axes
        for ax in empty_axes[1:]:
            ax.axis("off")
    else:
        # no empty panels — place legend above the figure
        fig.legend(
            handles=legend_handles,
            loc="upper center",
            bbox_to_anchor=(0.5, 1.02),
            ncol=len(glaciers_to_plot),
            fontsize=8,
            frameon=False,
        )

    plt.tight_layout(h_pad=3.0)
    plt.savefig(f"figures/paperTF/{save_prefix}_kde.pdf", bbox_inches="tight")
    plt.savefig(f"figures/paperTF/{save_prefix}_kde.png",
                dpi=300,
                bbox_inches="tight")
    plt.show()

In [ ]:
TYPE = 'JOINT'
if TYPE == 'JOINT':
    ISL_FT_POOL = ISL_FT_POOL_JOINT
    ISL_TEST_SET = ISL_TEST_SET_JOINT
    df_isl_pool = df_isl_pool_joint
    df_isl_test = df_isl_test_joint
else:
    ISL_FT_POOL = ISL_FT_POOL_TOPO
    ISL_TEST_SET = ISL_TEST_SET_TOPO
    df_isl_pool = df_isl_pool_topo
    df_isl_test = df_isl_test_topo

In [ ]:
selected_cols = MONTHLY_COLS + [
    c for c in STATIC_COLS if c != "ELEVATION_DIFFERENCE"
]

# SELECTED_COLS = [
#     't2m',
#     'tp',
#     'ssrd',
#     'ELEVATION_DIFFERENCE',
#     'aspect',
#     'slope',
# ]

glaciers_to_plot = {
    f"Full ISL ({df_isl_all['GLACIER'].nunique()} glaciers)": {
        "df": df_isl_all,
        "color": "#4dac26",  # green
    },
    f"FT pool ({len(ISL_FT_POOL)} glaciers)": {
        "df": df_isl_pool,
        "color": "#2166ac",  # blue
    },
    f"Test set ({len(ISL_TEST_SET)} glaciers)": {
        "df": df_isl_test,
        "color": "#b2182b",  # red
    },
}

plot_kde_pair(
    glaciers_to_plot=glaciers_to_plot,
    selected_cols=selected_cols,
    save_prefix="isl_pool_vs_test",
)

#### FT Pool subsets:

In [ ]:
# ── Step B: Generate all FT subsets (informed joint + topo + random) ──────

import json
from pathlib import Path

FRACS        = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
N_SEEDS      = 10
N_TOTAL_ISL  = len(df_isl_all)

SUBSETS_PATH       = BASE_DIR / "isl_ft_subsets_glacier.json"
RECOMPUTE_JOINT    = True
RECOMPUTE_TOPO     = True
RECOMPUTE_RANDOM   = True

# ── Load or init ──────────────────────────────────────────────────────────
if SUBSETS_PATH.exists() and not any([RECOMPUTE_JOINT, RECOMPUTE_TOPO, RECOMPUTE_RANDOM]):
    with open(SUBSETS_PATH) as f:
        ft_subsets = json.load(f)
    print(f"Loaded FT subsets from {SUBSETS_PATH}")
else:
    ft_subsets = {"informed_joint": {}, "informed_topo": {}, "random": {}}

# ── Informed joint splits ─────────────────────────────────────────────────
if RECOMPUTE_JOINT or "informed_joint" not in ft_subsets:
    ft_subsets["informed_joint"] = {}
    for frac in FRACS:
        pct = int(frac * 100)
        target_meas       = int(round(frac * N_TOTAL_ISL))
        pool_frac_relative = target_meas / len(df_isl_pool)

        if pool_frac_relative >= 1.0:
            print(f"  Warning: {pct}% of total exceeds pool size, capping.")
            pool_frac_relative = 0.99

        print(f"\nInformed JOINT @ {pct}%  ({target_meas} meas, "
              f"pool_frac_relative={pool_frac_relative:.3f})")

        result = split_pool_holdout_sinkhorn_v2(
            df_region=df_isl_pool,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            scaler_m=scaler_m,
            scaler_s=scaler_s,
            blur_joint=blur_joint,
            mode="joint",
            pool_frac=pool_frac_relative,
            n_restarts=50,
            seed=cfg.seed,
        )

        ft_subsets["informed_joint"][str(pct)] = {
            "glaciers":   result["pool_glaciers"],
            "n_glaciers": len(result["pool_glaciers"]),
            "n_meas":     result["n_meas_pool"],
            "actual_frac": result["n_meas_pool"] / N_TOTAL_ISL,
            "score":      result["best_score"],
        }
        print(f"  → {len(result['pool_glaciers'])} glaciers | "
              f"{result['n_meas_pool']} meas "
              f"({result['n_meas_pool']/N_TOTAL_ISL:.1%}) | "
              f"score={result['best_score']:.4f}")

# ── Informed topo splits ──────────────────────────────────────────────────
if RECOMPUTE_TOPO or "informed_topo" not in ft_subsets:
    ft_subsets["informed_topo"] = {}
    for frac in FRACS:
        pct = int(frac * 100)
        target_meas        = int(round(frac * N_TOTAL_ISL))
        pool_frac_relative = target_meas / len(df_isl_pool)

        if pool_frac_relative >= 1.0:
            print(f"  Warning: {pct}% of total exceeds pool size, capping.")
            pool_frac_relative = 0.99

        print(f"\nInformed TOPO @ {pct}%  ({target_meas} meas, "
              f"pool_frac_relative={pool_frac_relative:.3f})")

        result = split_pool_holdout_sinkhorn_v2(
            df_region=df_isl_pool,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            scaler_m=scaler_m,
            scaler_s=scaler_s,
            blur_joint=blur_joint,
            mode="topo",
            topo_extra_cols=["ELEVATION_DIFFERENCE"],
            pool_frac=pool_frac_relative,
            n_restarts=50,
            seed=cfg.seed,
        )

        ft_subsets["informed_topo"][str(pct)] = {
            "glaciers":   result["pool_glaciers"],
            "n_glaciers": len(result["pool_glaciers"]),
            "n_meas":     result["n_meas_pool"],
            "actual_frac": result["n_meas_pool"] / N_TOTAL_ISL,
            "score":      result["best_score"],
        }
        print(f"  → {len(result['pool_glaciers'])} glaciers | "
              f"{result['n_meas_pool']} meas "
              f"({result['n_meas_pool']/N_TOTAL_ISL:.1%}) | "
              f"score={result['best_score']:.4f}")

# ── Random splits ─────────────────────────────────────────────────────────
if RECOMPUTE_RANDOM or "random" not in ft_subsets:
    ft_subsets["random"] = {}
    rng = np.random.default_rng(cfg.seed)

    for frac in FRACS:
        pct         = int(frac * 100)
        target_meas = int(round(frac * N_TOTAL_ISL))
        ft_subsets["random"][str(pct)] = []

        # cap at informed_joint n_meas for fair comparison
        informed_n_meas = ft_subsets["informed_joint"][str(pct)]["n_meas"]
        g_counts        = df_isl_pool.groupby("GLACIER").size().loc[ISL_FT_POOL]

        print(f"\nRandom @ {pct}%  (target {target_meas} meas, "
              f"capped at {informed_n_meas} meas)")

        for seed_idx in range(N_SEEDS):
            seed      = int(rng.integers(0, 2**31))
            rng_local = np.random.default_rng(seed)

            # Step 1: shuffle glaciers, greedily accumulate until target_meas
            shuffled = list(rng_local.permutation(ISL_FT_POOL))
            selected_glaciers, n_meas_selected = [], 0
            for g in shuffled:
                selected_glaciers.append(g)
                n_meas_selected += int(g_counts[g])
                if n_meas_selected >= target_meas:
                    break

            # Step 2: trim at ID level to match informed_n_meas
            df_selected = df_isl_pool[df_isl_pool["GLACIER"].isin(selected_glaciers)]
            if n_meas_selected > informed_n_meas:
                all_ids = list(df_selected["ID"].unique())
                rng_local.shuffle(all_ids)
                kept_ids, n_kept = [], 0
                for id_ in all_ids:
                    id_meas = int((df_selected["ID"] == id_).sum())
                    if n_kept + id_meas <= informed_n_meas:
                        kept_ids.append(id_)
                        n_kept += id_meas
                selected_glaciers = list(
                    df_selected[df_selected["ID"].isin(kept_ids)]["GLACIER"].unique()
                )
                n_meas_selected = n_kept

            ft_subsets["random"][str(pct)].append({
                "seed_idx":    seed_idx,
                "seed":        seed,
                "glaciers":    selected_glaciers,
                "n_glaciers":  len(selected_glaciers),
                "n_meas":      n_meas_selected,
                "actual_frac": n_meas_selected / N_TOTAL_ISL,
            })

        actual_fracs = [s["actual_frac"] for s in ft_subsets["random"][str(pct)]]
        print(f"  → avg {sum(s['n_glaciers'] for s in ft_subsets['random'][str(pct)])/N_SEEDS:.1f} glaciers | "
              f"frac range [{min(actual_fracs):.1%} – {max(actual_fracs):.1%}]")

# ── Save ──────────────────────────────────────────────────────────────────
if any([RECOMPUTE_JOINT, RECOMPUTE_TOPO, RECOMPUTE_RANDOM]):
    with open(SUBSETS_PATH, "w") as f:
        json.dump(ft_subsets, f, indent=2)
    print(f"\nSubsets saved to {SUBSETS_PATH}")

# ── Sanity check ──────────────────────────────────────────────────────────
print("\nSubset summary:")
print(f"{'':6} {'informed_joint':>22} {'informed_topo':>22} {'random (mean)':>25}")
print(f"{'frac':6} {'glac':>6} {'meas':>7} {'act%':>6}  "
      f"{'glac':>6} {'meas':>7} {'act%':>6}  "
      f"{'glac':>6} {'meas':>7} {'act%':>6}")
for frac in FRACS:
    pct = str(int(frac * 100))
    j   = ft_subsets["informed_joint"][pct]
    t   = ft_subsets["informed_topo"][pct]
    rnd = ft_subsets["random"][pct]
    rnd_frac = sum(s["actual_frac"] for s in rnd) / N_SEEDS
    print(
        f"{int(pct):>4}%  "
        f"{j['n_glaciers']:>6}  {j['n_meas']:>7}  {j['actual_frac']:>5.1%}  "
        f"{t['n_glaciers']:>6}  {t['n_meas']:>7}  {t['actual_frac']:>5.1%}  "
        f"{sum(s['n_glaciers'] for s in rnd)/N_SEEDS:>6.1f}  "
        f"{sum(s['n_meas'] for s in rnd)/N_SEEDS:>7.0f}  {rnd_frac:>5.1%}"
    )

# ── Glacier overlap between joint and topo selections ────────────────────
print("\nGlacier overlap (joint vs topo informed):")
for frac in FRACS:
    pct = str(int(frac * 100))
    j_set = set(ft_subsets["informed_joint"][pct]["glaciers"])
    t_set = set(ft_subsets["informed_topo"][pct]["glaciers"])
    jaccard = len(j_set & t_set) / len(j_set | t_set) if j_set | t_set else 0
    print(f"  {int(pct):>3}%  overlap={len(j_set & t_set)}/{len(j_set | t_set)}  "
          f"Jaccard={jaccard:.3f}  "
          f"joint_only={sorted(j_set - t_set)}  "
          f"topo_only={sorted(t_set - j_set)}")

In [ ]:
# ── KDE: visualise a specific FT subset ───────────────────────────────────

pct      = "20"
strategy = "informed_topo"  # "informed_joint" | "informed_topo" | "random"
seed_idx = 0                 # only relevant for random

if strategy == "informed_joint":
    subset_glaciers = ft_subsets["informed_joint"][pct]["glaciers"]
    label = (f"Informed joint {pct}% "
             f"({len(subset_glaciers)} glaciers, "
             f"score={ft_subsets['informed_joint'][pct]['score']:.4f})")
    color = "#b2182b"
    save_suffix = f"informed_joint_{pct}pct"

elif strategy == "informed_topo":
    subset_glaciers = ft_subsets["informed_topo"][pct]["glaciers"]
    label = (f"Informed topo {pct}% "
             f"({len(subset_glaciers)} glaciers, "
             f"score={ft_subsets['informed_topo'][pct]['score']:.4f})")
    color = "#e08214"
    save_suffix = f"informed_topo_{pct}pct"

else:  # random
    s = ft_subsets["random"][pct][seed_idx]
    subset_glaciers = s["glaciers"]
    label = f"Random {pct}% seed{seed_idx} ({len(subset_glaciers)} glaciers)"
    color = "#762a83"
    save_suffix = f"random_{pct}pct_seed{seed_idx}"

df_subset = df_isl_pool[df_isl_pool["GLACIER"].isin(subset_glaciers)]

glaciers_to_plot = {
    f"Full ISL ({df_isl_all['GLACIER'].nunique()} glaciers)": {
        "df":    df_isl_all,
        "color": "#4dac26",
    },
    f"FT pool ({len(ISL_FT_POOL)} glaciers)": {
        "df":    df_isl_pool,
        "color": "#2166ac",
    },
    label: {
        "df":    df_subset,
        "color": color,
    },
}

plot_kde_pair(
    glaciers_to_plot=glaciers_to_plot,
    selected_cols=selected_cols,
    save_prefix=f"isl_{save_suffix}",
)

# ── Quick summary of the selected subset ─────────────────────────────────
print(f"Strategy : {strategy}")
print(f"Fraction : {pct}%")
print(f"Glaciers : {subset_glaciers}")
print(f"N rows   : {len(df_subset)}")
print(f"N IDs    : {df_subset['ID'].nunique()}")

### Plot FT/Hold-out glaciers:

##### Iceland:

In [ ]:
# ── Map: pool/test split + FT subsets ────────────────────────────────────

pct      = "20"
seed_idx = 0   # for random

# ── 1. Pool / test split (same as before) ────────────────────────────────
ft_glaciers_by_split     = {"exp_pool": ISL_FT_POOL}
holdout_glaciers_by_split = {"exp_pool": ISL_TEST_SET}

data_ISL, glacier_outline_rgi, glacier_info_by_split = build_region_glacier_info_for_splits(
    cfg,
    rgi_region_id="06",
    outline_shp_path=cfg.dataPath + "RGI_v6/RGI_06_Iceland/06_rgi60_Iceland.shp",
    ft_glaciers_by_split=ft_glaciers_by_split,
    split_names=("exp_pool",),
    ft_label_col="FT/Hold-out glacier",
    holdout_glaciers_by_split=holdout_glaciers_by_split,
)

glacier_df_ISL_exp = glacier_info_by_split["exp_pool"]
train_color = get_cmap_hex(cm.batlow, 10)[0]
palette = {"Hold-out": train_color, "FT": "#2166ac"}

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_ISL_exp,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Iceland — FT pool / test split",
    extent=(-25, -11, 62, 68),
    sizes=(100, 1500),
    size_legend_values=(30, 100, 1000),
    palette=palette,
    cmap_for_train=cm.batlow,
    split_col="FT/Hold-out glacier",
)

# ── 2. FT subset maps — one per strategy ─────────────────────────────────
strategies = {
    "informed_joint": {
        "glaciers": ft_subsets["informed_joint"][pct]["glaciers"],
        "color_ft": "#b2182b",
        "title":    f"Iceland — Informed joint {pct}% FT subset "
                    f"({ft_subsets['informed_joint'][pct]['n_glaciers']} glaciers)",
    },
    "informed_topo": {
        "glaciers": ft_subsets["informed_topo"][pct]["glaciers"],
        "color_ft": "#e08214",
        "title":    f"Iceland — Informed topo {pct}% FT subset "
                    f"({ft_subsets['informed_topo'][pct]['n_glaciers']} glaciers)",
    },
    "random": {
        "glaciers": ft_subsets["random"][pct][seed_idx]["glaciers"],
        "color_ft": "#762a83",
        "title":    f"Iceland — Random {pct}% FT subset seed{seed_idx} "
                    f"({ft_subsets['random'][pct][seed_idx]['n_glaciers']} glaciers)",
    },
}

for strategy_name, cfg_s in strategies.items():
    # selected = FT subset, unselected pool glaciers = hold-out for this map
    pool_not_selected = [
        g for g in ISL_FT_POOL if g not in cfg_s["glaciers"]
    ]

    ft_by_split_s      = {"subset": cfg_s["glaciers"]}
    holdout_by_split_s = {"subset": pool_not_selected + ISL_TEST_SET}

    _, _, glacier_info_by_split_s = build_region_glacier_info_for_splits(
        cfg,
        rgi_region_id="06",
        outline_shp_path=cfg.dataPath + "RGI_v6/RGI_06_Iceland/06_rgi60_Iceland.shp",
        ft_glaciers_by_split=ft_by_split_s,
        split_names=("subset",),
        ft_label_col="FT/Hold-out glacier",
        holdout_glaciers_by_split=holdout_by_split_s,
    )

    glacier_df_s = glacier_info_by_split_s["subset"]
    palette_s    = {"Hold-out": train_color, "FT": cfg_s["color_ft"]}

    fig, ax, _, _ = plot_glacier_measurements_map(
        glacier_info=glacier_df_s,
        glacier_outline_rgi=glacier_outline_rgi,
        title=cfg_s["title"],
        extent=(-25, -11, 62, 68),
        sizes=(100, 1500),
        size_legend_values=(30, 100, 1000),
        palette=palette_s,
        cmap_for_train=cm.batlow,
        split_col="FT/Hold-out glacier",
    )
    plt.savefig(
        f"figures/paperTF/isl_map_{strategy_name}_{pct}pct.pdf",
        bbox_inches="tight",
    )
    plt.savefig(
        f"figures/paperTF/isl_map_{strategy_name}_{pct}pct.png",
        dpi=150, bbox_inches="tight",
    )
    plt.show()

## LSTM model
### LSTM datasets:

In [ ]:
# ── Step C: Build TL assets for CH→ISL experiment ────────────────────────
# Strategies: informed_joint, informed_topo, random
# Total: 6 fracs × 3 strategies × (1 informed or 10 random seeds)
#      = 6 × 1 + 6 × 1 + 6 × 10 = 72 subsets

EXP_FT_GLACIERS = {}

for frac in FRACS:
    pct = int(frac * 100)

    # Informed joint
    key = f"ISL_informed_joint_{pct}pct"
    EXP_FT_GLACIERS[key] = ft_subsets["informed_joint"][str(pct)]["glaciers"]

    # Informed topo
    key = f"ISL_informed_topo_{pct}pct"
    EXP_FT_GLACIERS[key] = ft_subsets["informed_topo"][str(pct)]["glaciers"]

    # Random seeds
    for s in ft_subsets["random"][str(pct)]:
        key = f"ISL_random_{pct}pct_seed{s['seed_idx']}"
        EXP_FT_GLACIERS[key] = s["glaciers"]

print(f"Total FT subsets: {len(EXP_FT_GLACIERS)}")
print("Keys:", list(EXP_FT_GLACIERS.keys()))

# build_transfer_learning_assets expects:
# { target_region: { split_label: [glacier_list] } }
FT_GLACIERS_EXP = {"ISL": {k: v for k, v in EXP_FT_GLACIERS.items()}}

tl_assets_exp = build_transfer_learning_assets(
    cfg=cfg,
    res_xreg=res_xreg_by_source["CH"],
    FT_GLACIERS=FT_GLACIERS_EXP,
    MONTHLY_COLS=MONTHLY_COLS,
    STATIC_COLS=STATIC_COLS,
    cache_dir=BASE_DIR / "LSTM_cache_TL" / "exp_ch_isl_glacier",
    force_recompute=False,
    src_code="CH",
    region_groups={},
)

print(f"\nTL asset keys ({len(tl_assets_exp)}):")
for k in tl_assets_exp:
    print(f"  {k}")

# ── Sanity check: verify counts per strategy ──────────────────────────────
from collections import Counter
strategy_counts = Counter()
for k in tl_assets_exp:
    if "informed_joint" in k:
        strategy_counts["informed_joint"] += 1
    elif "informed_topo" in k:
        strategy_counts["informed_topo"] += 1
    elif "random" in k:
        strategy_counts["random"] += 1

print(f"\nAssets per strategy:")
for s, c in strategy_counts.items():
    print(f"  {s:20}: {c}")
print(f"  {'TOTAL':20}: {sum(strategy_counts.values())}")

### LSTM parameters:

In [ ]:
log_path_gs_results = {
    "ISL":
    GLOB_DIR / 'GS_results/lstm_param_search_progress_OOS_ISL_2026-02-11.csv',
    "NOR":
    GLOB_DIR / 'GS_results/lstm_param_search_progress_OOS_NOR_2026-02-09.csv',
    "FR":
    GLOB_DIR / 'GS_results/lstm_param_search_progress_OOS_FR_2026-02-06.csv',
    "CH": GLOB_DIR / 'GS_results/lstm_param_search_progress_CH_2026-02-18.csv',
}

default_params = {
    'Fm': 8,
    'Fs': 3,
    'hidden_size': 128,
    'num_layers': 1,
    'bidirectional': False,
    'dropout': 0.0,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0.1,
    'lr': 0.001,
    'weight_decay': 1e-05,
    'loss_name': 'neutral',
    'two_heads': False,
    'head_dropout': 0.1,
    'loss_spec': None
}

params_by_key = build_lstm_params_by_key(
    default_params=default_params,
    log_path_gs_results=log_path_gs_results,
    RGI_REGIONS=RGI_REGIONS,
)


def _params_key_for_source(src_region, params_by_key, default_params):
    """Look up best params for a source region using RGI_REGIONS mapping."""
    for rgi, info in RGI_REGIONS.items():
        if info["code"] == src_region:
            key = f"{rgi}_{src_region}"
            return params_by_key.get(key, default_params)
    # CEU has no single RGI — fall back to default
    return default_params


params_by_key.keys()

### Train or load CH baseline:

In [ ]:
# ── Step D: Train or load CH baseline ────────────────────────────────────
# We only need CH as the source for this experiment.

TRAIN_FLAG = True  # set True to retrain from scratch
MODEL_DATE = "2026-05-20"

ch_baseline_model, ch_baseline_path, ch_baseline_info = train_or_load_CH_baseline(
    cfg=cfg,
    tl_assets=
    tl_assets_exp,  # use exp assets — any key works, just needs the train split
    default_params=_params_key_for_source("CH", params_by_key, default_params),
    device=device,
    models_dir=BASE_DIR / "models" / "CH",
    prefix="lstm_CH",
    key="defaultparams",
    train_flag=TRAIN_FLAG,
    force_retrain=False,
    epochs=150,
    verbose=False,
    date=MODEL_DATE,
)

print(f"CH baseline loaded from: {ch_baseline_path}")

### Fine tune LSTM:

### Adapters:

In [ ]:
# load GS results:
import json

with open(
        GLOB_DIR /
        'GS_results/tuning_adapter/adapter_best_by_region_2026-02-24_15.json',
        "r") as f:
    best_by_region = json.load(f)

# ── Step E: Fine-tune with adapters for all CH→ISL subsets ───────────────

FORCE_RETRAIN = True
FT_DATE = "2026-05-20"

exp_adapter_models = {}
exp_adapter_infos = {}

exp_adapter_models, exp_adapter_infos = finetune_TL_models_all(
    cfg=cfg,
    tl_assets_by_key=tl_assets_exp,
    best_params=_params_key_for_source("CH", params_by_key, default_params),
    device=device,
    pretrained_ckpt_path=ch_baseline_path,
    strategies=["adapter"],
    force_retrain=FORCE_RETRAIN,
    prefix="lstm_CH_to_ISL_exp",
    verbose=False,
    best_by_region=best_by_region,
    models_dir=BASE_DIR / "models" / "CH" / "exp_ch_isl",
    date=FT_DATE,
)

print(f"Fine-tuned model keys ({len(exp_adapter_models)}):")
for k in exp_adapter_models:
    print(f"  {k}")

### DAN:

In [ ]:
# ── Step E2: Fine-tune with DAN for all CH→ISL subsets ───────────────────

FORCE_RETRAIN_DAN = False
DAN_DATE = "2026-05-11"

# DAN's regions_only expects the target region labels as they appear in the
# asset keys — extract them the same way the original code does
dan_regions = sorted(
    set(
        k.split("TL_CH_to_")[1].rsplit("_", 1)[0]
        for k in tl_assets_exp.keys()))
print(f"DAN target regions: {dan_regions}")

exp_dan_models, exp_dan_infos = finetune_TL_models_all(
    cfg=cfg,
    tl_assets_by_key=tl_assets_exp,
    best_params=_params_key_for_source("CH", params_by_key, default_params),
    device=device,
    pretrained_ckpt_path=ch_baseline_path,
    strategies=["dan"],
    force_retrain=FORCE_RETRAIN_DAN,
    prefix="lstm_CH_to_ISL_exp",
    verbose=False,
    regions_only=dan_regions,
    best_by_region=best_by_region,
    models_dir=BASE_DIR / "models" / "CH" / "exp_ch_isl_2",
    date=DAN_DATE,
)

print(f"DAN model keys ({len(exp_dan_models)}):")
for k in exp_dan_models:
    print(f"  {k}")

### Evaluate on test:

In [ ]:
# ── Step F: Evaluate all fine-tuned models on fixed ISL test set ──────────

results = []

for frac in FRACS:
    pct = int(frac * 100)

    # ── Informed joint ────────────────────────────────────────────────────
    asset_key = f"TL_CH_to_ISL_ISL_informed_joint_{pct}pct"
    model_key = [k for k in exp_adapter_models if f"informed_joint_{pct}pct" in k]

    if not model_key:
        print(f"WARNING: no model found for informed_joint_{pct}pct, skipping")
    else:
        ax = plt.subplot(1, 1, 1); plt.close()
        metrics, df_preds, _, _ = evaluate_one_model_TL(
            cfg=cfg,
            model=exp_adapter_models[model_key[0]],
            device=device,
            tl_assets_for_key=tl_assets_exp[asset_key],
            ax=ax, ax_xlim=None, ax_ylim=None,
            title=None, legend_fontsize=10,
            batch_size=128, domain_vocab=None, show_legend=False,
        )
        results.append({
            "strategy":   "informed_joint",
            "frac":       frac,
            "pct":        pct,
            "seed_idx":   None,
            "n_glaciers": ft_subsets["informed_joint"][str(pct)]["n_glaciers"],
            "n_meas":     ft_subsets["informed_joint"][str(pct)]["n_meas"],
            "score":      ft_subsets["informed_joint"][str(pct)]["score"],
            **metrics,
        })
        print(f"  informed_joint @ {pct}%: {metrics}")

    # ── Informed topo ─────────────────────────────────────────────────────
    asset_key = f"TL_CH_to_ISL_ISL_informed_topo_{pct}pct"
    model_key = [k for k in exp_adapter_models if f"informed_topo_{pct}pct" in k]

    if not model_key:
        print(f"WARNING: no model found for informed_topo_{pct}pct, skipping")
    else:
        ax = plt.subplot(1, 1, 1); plt.close()
        metrics, df_preds, _, _ = evaluate_one_model_TL(
            cfg=cfg,
            model=exp_adapter_models[model_key[0]],
            device=device,
            tl_assets_for_key=tl_assets_exp[asset_key],
            ax=ax, ax_xlim=None, ax_ylim=None,
            title=None, legend_fontsize=10,
            batch_size=128, domain_vocab=None, show_legend=False,
        )
        results.append({
            "strategy":   "informed_topo",
            "frac":       frac,
            "pct":        pct,
            "seed_idx":   None,
            "n_glaciers": ft_subsets["informed_topo"][str(pct)]["n_glaciers"],
            "n_meas":     ft_subsets["informed_topo"][str(pct)]["n_meas"],
            "score":      ft_subsets["informed_topo"][str(pct)]["score"],
            **metrics,
        })
        print(f"  informed_topo @ {pct}%: {metrics}")

    # ── Random seeds ──────────────────────────────────────────────────────
    for seed_idx in range(N_SEEDS):
        asset_key = f"TL_CH_to_ISL_ISL_random_{pct}pct_seed{seed_idx}"
        model_key = [
            k for k in exp_adapter_models
            if f"random_{pct}pct_seed{seed_idx}" in k
        ]

        if not model_key:
            print(f"WARNING: no model found for random_{pct}pct_seed{seed_idx}, skipping")
            continue

        ax = plt.subplot(1, 1, 1); plt.close()
        metrics, df_preds, _, _ = evaluate_one_model_TL(
            cfg=cfg,
            model=exp_adapter_models[model_key[0]],
            device=device,
            tl_assets_for_key=tl_assets_exp[asset_key],
            ax=ax, ax_xlim=None, ax_ylim=None,
            title=None, legend_fontsize=10,
            batch_size=128, domain_vocab=None, show_legend=False,
        )
        results.append({
            "strategy":   "random",
            "frac":       frac,
            "pct":        pct,
            "seed_idx":   seed_idx,
            "n_glaciers": ft_subsets["random"][str(pct)][seed_idx]["n_glaciers"],
            "n_meas":     ft_subsets["random"][str(pct)][seed_idx]["n_meas"],
            "score":      None,
            **metrics,
        })

    print(f"  random @ {pct}%: done ({N_SEEDS} seeds)")

# ── Save & summarise ──────────────────────────────────────────────────────
df_results = pd.DataFrame(results)
df_results.to_csv(BASE_DIR / "exp_ch_isl_results_glacier.csv", index=False)
print(f"\nResults saved — {len(df_results)} rows")
print(df_results.groupby(["strategy", "pct"]).mean(numeric_only=True).round(3))

In [ ]:
# ── Evaluate CH baseline (no FT) on ISL test set ─────────────────────────
ax_dummy = plt.subplot(1, 1, 1)
plt.close()

# use any asset key — all share the same ISL test set
any_isl_key = next(k for k in tl_assets_exp if "informed_joint_5pct" in k)

baseline_metrics, _, _, _ = evaluate_one_model_TL(
    cfg=cfg,
    model=ch_baseline_model,
    device=device,
    tl_assets_for_key=tl_assets_exp[any_isl_key],
    ax=ax_dummy,
    ax_xlim=None,
    ax_ylim=None,
    title=None,
    legend_fontsize=10,
    batch_size=128,
    domain_vocab=None,
    show_legend=False,
)
print("CH baseline (no FT) metrics:", baseline_metrics)

In [ ]:
import matplotlib.ticker as mtick

# ── Step G2: Combined plot — informed (joint + topo) overlaid on random ───

METRICS    = ["RMSE_annual", "RMSE_winter", "R2_annual", "R2_winter"]
FRACS_PLOT = sorted(df_results["frac"].unique())
PCTS_PLOT  = [int(f * 100) for f in FRACS_PLOT]

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(12, 8), sharex=True)
axes = axes.flatten()

for ax, metric in zip(axes, METRICS):
    is_r2 = metric.startswith("R2")

    # ── Random: mean ± std band + individual seed dots ───────────────────
    df_rnd = df_results[df_results["strategy"] == "random"]
    agg = df_rnd.groupby("pct")[metric].agg(["mean", "std"]).reindex(PCTS_PLOT)

    ax.plot(PCTS_PLOT, agg["mean"], marker="o", color="steelblue",
            label="Random (mean)", zorder=4)
    ax.fill_between(
        PCTS_PLOT,
        agg["mean"] - agg["std"],
        agg["mean"] + agg["std"],
        alpha=0.2, color="steelblue", label="Random ±1 std",
    )
    for _, row_r in df_rnd.iterrows():
        ax.scatter(row_r["pct"], row_r[metric],
                   color="steelblue", alpha=0.25, s=15, zorder=3)

    mean_rnd = agg["mean"].values
    x = np.array(PCTS_PLOT)

    # ── Informed joint ────────────────────────────────────────────────────
    df_inf_j   = df_results[df_results["strategy"] == "informed_joint"]
    inf_j_vals = df_inf_j.groupby("pct")[metric].mean().reindex(PCTS_PLOT)
    inf_j_arr  = inf_j_vals.values

    ax.plot(PCTS_PLOT, inf_j_vals, marker="D", linestyle="--",
            color="darkorange", label="Informed joint (Sinkhorn)",
            zorder=5, markersize=8)

    better_j = inf_j_arr < mean_rnd if not is_r2 else inf_j_arr > mean_rnd
    ax.fill_between(x, mean_rnd, inf_j_arr,
                    where=better_j, alpha=0.12, color="green")
    ax.fill_between(x, mean_rnd, inf_j_arr,
                    where=~better_j, alpha=0.12, color="red")

    # ── Informed topo ─────────────────────────────────────────────────────
    df_inf_t   = df_results[df_results["strategy"] == "informed_topo"]
    inf_t_vals = df_inf_t.groupby("pct")[metric].mean().reindex(PCTS_PLOT)
    inf_t_arr  = inf_t_vals.values

    ax.plot(PCTS_PLOT, inf_t_vals, marker="s", linestyle="-.",
            color="#e08214", label="Informed topo (Sinkhorn)",
            zorder=5, markersize=8)

    better_t = inf_t_arr < mean_rnd if not is_r2 else inf_t_arr > mean_rnd
    ax.fill_between(x, mean_rnd, inf_t_arr,
                    where=better_t, alpha=0.08, color="green",
                    hatch="//", linewidth=0)
    ax.fill_between(x, mean_rnd, inf_t_arr,
                    where=~better_t, alpha=0.08, color="red",
                    hatch="//", linewidth=0)

    # ── CH baseline (no FT) ───────────────────────────────────────────────
    ax.axhline(baseline_metrics[metric], color="grey", linestyle=":",
               linewidth=1.5, label="CH baseline (no FT)", zorder=2)

    # ── Formatting ────────────────────────────────────────────────────────
    ax.set_title(metric, fontsize=11)
    ax.set_ylabel(metric, fontsize=10)
    ax.set_xlabel("FT data (% of total ISL)", fontsize=10)
    ax.set_xticks(PCTS_PLOT)
    ax.xaxis.set_major_formatter(mtick.FormatStrFormatter("%d%%"))
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, framealpha=0.9)

fig.suptitle(
    "CH → ISL  |  Adapter fine-tuning: Informed (joint & topo) vs. Random\n"
    f"Fixed test set: {len(ISL_TEST_SET)} glaciers | {N_SEEDS} random seeds",
    fontsize=13,
)
plt.tight_layout()
fig.savefig(BASE_DIR / "exp_ch_isl_combined_glacier.pdf", bbox_inches="tight")
fig.savefig(BASE_DIR / "exp_ch_isl_combined_glacier.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")